# 第 33 章：风险管理（Risk Management）

本 Notebook 从核心模块与章节 CLI 重算风险结果：先看离散分位数与分数尾部，再比较同一组合的历史、正态、Student-t 与 Monte Carlo 估计，随后检查第 29 章账本持仓的压力重估、退出时间、保证金与限额回传。最后对比构造数据和 French 行业月收益的风险模型回测。

损失定义为负的组合价值变化，正值表示亏损。每个结果都同时给出置信水平和持有区间。模拟数据来自脚本中公开的已知数据生成过程；压力结果是条件情景，不是概率预测。Notebook 只读取数据、计算结果并展示已生成图形，不写出实验文件。

**I2 默认离线路线：CONSTRUCTED 教学计算。** French 数据未随包提供，权利/用途 unknown；真实市场分支显式跳过，不能据此声称完整实证复现。原文历史数字只是章节背景，非本次输出。A10-A design accepted / execution blocked；A10-A.1 protocol technically accepted pending permissions and execution authorization；A10-B 未授权。静态图如显示则为已冻结教学参考，不表示本次重新生成。


In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch33_risk_management_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

import sys
from pathlib import Path

sys.path.insert(0, str(ROOT / "src"))
print("interpreter:", sys.executable)
print("project root:", ROOT)
import importlib.util


def load_code_module(filename: str):
    """Import a chapter CLI from code/ so the notebook calls exactly the functions the CLI runs."""
    spec = importlib.util.spec_from_file_location(Path(filename).stem, ROOT / "code" / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

## 1. 从核心函数重算 CLI 的结果

In [ ]:
import json
import numpy as np
from quant_course.risk import empirical_var_es

ch33 = load_code_module("ch33_risk_management.py")
results, arrays = ch33.analyze(include_french=False)
ch33.generate_figures(results, arrays, RUN_DIR)
print("章节：", results["title"])
print("损失符号与单位：", results["units_and_conventions"])
(RUN_DIR / "ch33_results.json").write_text(json.dumps(results, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


## 2. 离散 VaR、分位数积分与阈值上的分数尾部

In [ ]:
hand = results["pinned_discrete_case"]
inputs = results["teaching_inputs"]["pinned_discrete"]
counts = np.rint(np.asarray(inputs["probabilities"]) * hand["scenario_count"]).astype(int)
losses = np.repeat(inputs["loss_values"], counts)
var, es = empirical_var_es(losses, inputs["alpha"])
print("输入：", inputs)
print("VaR / ES：", var, es)
print("严格超越 VaR 的条件均值：", hand["strict_tail_conditional_mean"])
print("包含 VaR 点的条件均值：", hand["inclusive_tail_conditional_mean"])
assert np.isclose(var, hand["var"])
assert np.isclose(es, hand["es"])

## 3. 同一组合上的四种估计方法

In [ ]:
comparison = results["same_portfolio_risk_methods"]
for method in ("historical", "normal_fitted", "student_t_fitted", "monte_carlo"):
    print(method, comparison[method])
print("同方差尾部反例：", comparison["variance_matched_tail_counterexample"])

## 4. 第 29 章账本持仓、压力与风险限额

In [ ]:
stress = results["ledger_stress_and_limit"]
print("信号日 / 执行日：", stress["signal_date"], stress["execution_date"])
print("账本股份与隔夜情景：", stress["positions_at_signal_shares"], stress["overnight_gap_simple_returns"])
print("逐资产损益：", stress["pnl_by_asset_currency"])
print("流动性收缩：", stress["liquidity"])
print("构造保证金参数：", stress["margin"])
print("限额后的权重与实际成交：", stress["risk_limit"])

## 5. 回测估计窗、检验窗与滞后滚动分位数

In [ ]:
backtest = results["risk_model_backtest"]
print("固定模型估计窗：", backtest["estimation_window"])
print("样本外检验窗与 Kupiec POF：", backtest["test_window"])
print("零次和全超越边界：", backtest["boundary_count_checks"])
print("滚动分位数时序核对：", backtest["rolling_quantile_alignment"])
print("范围：", backtest["kupiec_test"])

## 6. French 行业月收益：真实修订数据的边界

In [ ]:
if RUN_FRENCH:
    real = results["french_real_example"]
    print({key: value for key, value in real.items() if key != "sha256"})
    print("SHA-256：", real["sha256"])
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


## 7. 展示 CLI 生成的四幅图

In [ ]:
from IPython.display import Image, display
for filename in ch33.FIGURES:
    display(Image(filename=str(RUN_DIR / filename)))